In [2]:
# Import necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')
import nltk


## Loading and inspecting

In [3]:
# Download required NLTK data
nltk.download('punkt')  # Sentence tokenizer
nltk.download('words')  # English word list
nltk.download('stopwords')  # Common stopwords
nltk.download('wordnet')  # WordNet lexical database
nltk.download('punkt_tab') # Download punkt_tab resource

In [4]:
# Import WordNet, a lexical database used for lemmatization and semantic analysis
from nltk.corpus import wordnet
# Import a list of common English stopwords (e.g., "the", "and", "is") used for filtering out non-informative words
from nltk.corpus import stopwords

import ftfy

In [5]:
bike_df = pd.read_csv(
    'bike_rental_reviews.csv',
    encoding='latin1'
)

bike_df['review_text'] = (
    bike_df['review_text']
    .astype(str)
    .apply(ftfy.fix_text)
)

car_df = pd.read_csv(
    'Car_Reviews_Database.csv',
    encoding='latin1'
)

car_df['Review'] = (
    car_df['Review']
    .astype(str)
    .apply(ftfy.fix_text)
)


## Data Collection & preprocessing

- Text cleaning (lowercasing, removing punctuation, stopword removal, lemmatization)
- Tokenization

In [ ]:
# check balance
bike_df['sentiment'].value_counts()

,count
sentiment,
negative,16840
positive,16777
neutral,16383


In [6]:
# Check for duplicates and drop if there are any
bike_df.drop_duplicates(inplace=True)
bike_df.shape

(300, 2)

In [7]:
# Get list of stopwords
stp_wrds_eng = stopwords.words('english')

def clean_my_text(df_text_col):
  # Lower casese
  clean_text = df_text_col.apply(lambda x: x.lower())
  # Remove punctuation
  clean_text = clean_text.apply(lambda x:re.sub(f'[{re.escape(string.punctuation)}]', '', x))
  # Tokenize reviews
  clean_text = clean_text.apply(lambda x: nltk.word_tokenize(x))
  # Lemmatize
  lemmatizer = nltk.stem.WordNetLemmatizer()
  clean_text = clean_text.apply(lambda x: [lemmatizer.lemmatize(word) for word in x])
  # Remove stopwords
  clean_text = clean_text.apply(lambda x: [word for word in x if word not in stp_wrds_eng])
  return clean_text



In [10]:
clean_text = clean_my_text(bike_df['review_text'])

bike_df['clean_text'] = clean_text
bike_df.head()

clean_text_car = clean_my_text(car_df['Review'])

car_df['clean_text'] = clean_text_car
car_df.head()

,Year,Model,Review,clean_text
0,2009,Honda,Although arguably the first-generation Insight...,"[although, arguably, firstgeneration, insight,..."
1,2009,Honda,2009 Honda Accord EX-L 4 : This car is very c...,"[2009, honda, accord, exl, 4, car, comfortable..."
2,2010,Honda,I have owed and driven Honda products for 20 y...,"[owed, driven, honda, product, 20, year, purch..."
3,2010,Honda,"Honda Accord Euro L : The seats are average, b...","[honda, accord, euro, l, seat, average, little..."
4,2011,Honda,Honda HR-V: Continuous variable transmission ...,"[honda, hrv, continuous, variable, transmissio..."


## Sentiment analysis


In [27]:
# Sentiment Analysis with TF-IDF, Naive Bayes, and Logistic Regression

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import StratifiedKFold

text_column = "clean_text"  # Column containing preprocessed review text
label_column = "sentiment" # Column with sentiment labels (e.g., 1, 0, -1)

# Split data using k-fold cross validation

bike_df.reset_index(drop=True, inplace=True) # Reset the index of bike_df after dropping duplicates

X = bike_df[text_column]        # Define features
y = bike_df[label_column]       # Define target

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_train, X_test, y_train, y_test = None, None, None, None   # Initialize variables to store the last fold's data

for train_idx, test_idx in skf.split(X, y):
    # Use .iloc for integer-location based indexing
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Join the lists of words into strings for TF-IDF vectorization
    X_train = X_train.apply(lambda x: ' '.join(x))
    X_test = X_test.apply(lambda x: ' '.join(x))


    # TF-IDF vectorization
    tfidf = TfidfVectorizer(max_features=5000)
    X_train_tfidf = tfidf.fit_transform(X_train)
    # Predict sentiment on bike data
    X_test_tfidf = tfidf.transform(X_test)

    # Naive Bayes Model
    nb = MultinomialNB()
    nb.fit(X_train_tfidf, y_train)
    nb_preds = nb.predict(X_test_tfidf)

    print("Naive Bayes Performance:")
    print("Accuracy:", accuracy_score(y_test, nb_preds))
    print(classification_report(y_test, nb_preds))

    # Logistic Regression Model
    logreg = LogisticRegression(max_iter=1000)
    logreg.fit(X_train_tfidf, y_train)
    logreg_preds = logreg.predict(X_test_tfidf)

    print("Logistic Regression Performance:")
    print("Accuracy:", accuracy_score(y_test, logreg_preds))
    print(classification_report(y_test, logreg_preds))

Naive Bayes Performance:
Accuracy: 1.0
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        20
     neutral       1.00      1.00      1.00        20
    positive       1.00      1.00      1.00        20

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60

Logistic Regression Performance:
Accuracy: 1.0
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        20
     neutral       1.00      1.00      1.00        20
    positive       1.00      1.00      1.00        20

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60

Naive Bayes Performance:
Accuracy: 1.0
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        20
    

In [26]:
# Shuffel the labels and rerun model to assess 100% accuracy result

from sklearn.utils import shuffle

X = bike_df[text_column].apply(lambda x: " ".join(x))
y = bike_df[label_column]

y_shuffled = shuffle(y, random_state=42).reset_index(drop=True)
X = X.reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y_shuffled), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    tfidf = TfidfVectorizer(max_features=5000)
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    logreg = LogisticRegression(max_iter=1000)
    logreg.fit(X_train_tfidf, y_train)
    preds = logreg.predict(X_test_tfidf)

    print(f"Fold {fold} — shuffled-label accuracy:", accuracy_score(y_test, preds))


Fold 1 — shuffled-label accuracy: 0.38333333333333336
Fold 2 — shuffled-label accuracy: 0.3333333333333333
Fold 3 — shuffled-label accuracy: 0.3
Fold 4 — shuffled-label accuracy: 0.26666666666666666
Fold 5 — shuffled-label accuracy: 0.2833333333333333


### Run BERT model
- On car data
- On Bike data

In [ ]:
from transformers import pipeline

# Load DistilBERT sentiment classifier
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)


labels = []
scores = []

for review in car_df['Review']:
    out = classifier(review, truncation=True)[0]
    labels.append(out['label'])
    scores.append(out['score'])

car_df['bert_sentiment'] = labels
car_df['bert_confidence'] = scores

# Change sentiment to 'neutral' if confidence is <0.6
def map_to_three_classes(label, score, threshold=0.6):
    if score < threshold:
        return "NEUTRAL"
    return label

car_df['sentiment_3class'] = [
    map_to_three_classes(l, s)
    for l, s in zip(car_df['bert_sentiment'], car_df['bert_confidence'])
]

car_df


Check BERT on labelled data to assess performance

In [ ]:
# Run sentiment analysis and collect results for bike_df
labels = []
scores = []

for review in bike_df['review_text']:
    # Return the prediction dictionary using [0]
    # Add truncation=True to handle sequences longer than the model's max length
    out = classifier(review, truncation=True)[0]
    labels.append(out['label'])
    scores.append(out['score'])

bike_df['bert_sentiment'] = labels
bike_df['bert_confidence'] = scores

bike_df['sentiment_3class'] = [
    map_to_three_classes(l, s)
    for l, s in zip(bike_df['bert_sentiment'], bike_df['bert_confidence'])
]

bike_df

### Project Overview
The objective of this project was to develop an end-to-end Natural Language Processing (NLP) pipeline to analyze customer review data and automatically classify sentiment. The task focused on extracting insights from unstructured textual feedback to understand customer satisfaction and dissatisfaction patterns. Both traditional machine learning models and transformer-based models were explored to compare performance and modeling trade-offs.
### Data Loading and Understanding
The customer review dataset was loaded into a Python environment and inspected to understand its structure and content. Key variables included the raw review text and a sentiment label (positive, neutral, negative). An initial class balance check showed that sentiment labels were relatively evenly distributed, making the dataset suitable for supervised sentiment classification.
### Text Preprocessing
To standardize the textual data and reduce noise, a preprocessing pipeline was implemented. The following steps were applied to the review text:
Conversion to lowercase
Removal of punctuation
Tokenization
Stopword removal
Lemmatization
The cleaned output was stored as a new column (clean_text). These steps ensured that downstream models focused on semantically meaningful tokens rather than stylistic or grammatical variations.
### Feature Engineering
The preprocessed text was converted into numerical features using Term Frequency–Inverse Document Frequency (TF-IDF). TF-IDF was chosen because it highlights words that are important to individual reviews while downweighting terms that appear frequently across all documents. The resulting TF-IDF matrix served as the input representation for traditional machine learning models.
### Sentiment Classification Models
Two supervised machine learning models were trained and evaluated:
Multinomial Naive Bayes
Logistic Regression
Both models were trained on TF-IDF features and evaluated using accuracy, precision, recall, and F1-score.
Observed Results
Both Naive Bayes and Logistic Regression achieved 100% accuracy, with perfect precision, recall, and F1-scores across all sentiment classes.

### Interpretation of Results

While NB and LR models achieved perfect perfoemance and accuracy, such results are highly unusual for real-world sentiment analysis tasks.

The unrealistically high performance strongly could also be due to potential data leakage in the dataset or preprocessing pipeline.

In sentiment analysis, common causes of such outcomes include:
- Labels being directly or indirectly encoded in the text
- Data leakage
- Overly clean or synthetic data

We therefore shiffled the labels and revealed that while our model was working correctly, the results were nearing chance level (0.33), meaning that the data is trivially separable. In practice, customer reviews are noisy, ambiguous, and often express mixed sentiment, making perfect separation unlikely.

Importantly, the identification of this issue is crucial for appropriate use if the method.

### Sentiment Analysis with BERT
In addition to traditional models, a pre-trained transformer model (BERT) was used to predict sentiment on a separate set of customer reviews. Unlike the supervised classifiers, this approach relied on a model fine-tuned on external sentiment data and was applied directly for inference.
The BERT model produced realistic sentiment predictions across positive, neutral, and negative classes, illustrating how transformer-based models can generalize sentiment understanding beyond the training dataset and avoid some pitfalls of dataset-specific leakage.

### Key Takeaways
A complete NLP pipeline was successfully implemented, covering preprocessing, feature extraction, and sentiment modeling.
Traditional TF-IDF–based classifiers performed very well, but results must be carefully validated to rule out leakage.

Perfect accuracy in sentiment analysis is a red flag rather than a success and should prompt further investigation.
Transformer-based models like BERT provide a robust alternative for sentiment inference, especially when labeled data quality is uncertain.

### Conclusion
This project demonstrated a full sentiment analysis workflow.  Beyond achieving high metrics, understanding why a model performs well is essential for building trustworthy NLP systems. The insights gained from this project informed the design of subsequent, more realistic NLP applications involving customer grievances and business decision-making, however, the key takeaway from this task is that the quality of data is the most important for correct and reliable modeling work.